# Natural Gradient Methods

| Difficulty | ████████░░ 8/10 |
| :--- | :--- |
| Prerequisites | 10.02 |
| Time | ~85 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/10_Differential_Geometry_and_Topology/03_natural_gradient_methods.ipynb)

---

## 🎯 Learning Objective

Implement natural gradient descent using the Fisher information matrix, and preview K-FAC as a scalable approximation.

---

## 📐 Theory

`10.02` established the Fisher information matrix $F(\theta)$ as the correct (Riemannian)
metric on a statistical manifold, and the natural gradient $F(\theta)^{-1}\nabla_\theta L$ as
steepest descent with respect to that metric rather than the arbitrary Euclidean one on raw
parameters. This notebook makes that algorithm concrete and confronts the two problems that
keep it from being used naively at neural-network scale: which Fisher matrix, exactly, and how
do you invert something with billions of entries.

### Natural gradient descent, as an algorithm

At each step: compute the gradient $g=\nabla_\theta L$, compute (or approximate) $F(\theta)$,
solve the linear system $F(\theta)\,\tilde g = g$ for $\tilde g$ (avoid forming $F^{-1}$
explicitly — solving a linear system is cheaper and more numerically stable than inverting), and
update $\theta \leftarrow \theta - \eta\,\tilde g$. This is identical in structure to Newton's
method (`04.07`) with $F$ in place of the Hessian $H$; the difference is entirely in *which*
matrix is used and why it's the theoretically correct choice for a model that outputs a
probability distribution.

### True Fisher vs. empirical Fisher

The **true Fisher matrix** uses the model's own predictive distribution to compute the
expectation: $F(\theta) = \mathbb{E}_{x\sim\text{data},\,y\sim p_\theta(\cdot|x)}[\nabla_\theta
\log p_\theta(y|x)\nabla_\theta \log p_\theta(y|x)^\top]$ — note $y$ is sampled from the *model*,
not from the real labels. The **empirical Fisher** substitutes the actual observed labels for
this inner sampling step, which is far cheaper (no need to sample from the model) but is only a
good approximation to the true Fisher near convergence, when the model's predictions and the
real data agree; away from convergence the two matrices can differ substantially, as we'll
verify numerically below rather than take on faith.

### K-FAC: exploiting the structure of a layer's gradient

Forming the full Fisher matrix for a neural network layer with weight matrix $W \in
\mathbb{R}^{d_{\text{out}}\times d_{\text{in}}}$ is intractable: $W$ has $d_{\text{out}}
d_{\text{in}}$ entries, so $F$ restricted to that one layer already has $(d_{\text{out}}
d_{\text{in}})^2$ entries. **K-FAC** (Kronecker-Factored Approximate Curvature; Martens & Grosse)
exploits a structural fact about backprop: the gradient with respect to $W$ for a single
example is the outer product $\nabla_W \ell = \delta \, a^\top$, where $a$ is the layer's input
activation and $\delta$ is the backpropagated error signal. K-FAC's approximation is to assume
$a$ and $\delta$ are independent, which lets the Fisher block for that layer factor as a
**Kronecker product** $F_{\text{layer}} \approx G \otimes A$, where $A = \mathbb{E}[aa^\top]$
($d_{\text{in}}\times d_{\text{in}}$) and $G = \mathbb{E}[\delta\delta^\top]$
($d_{\text{out}}\times d_{\text{out}}$) — two small matrices instead of one enormous one, and
Kronecker-product inverses factor too: $(G\otimes A)^{-1} = G^{-1}\otimes A^{-1}$, so you never
need to invert anything bigger than $d_{\text{in}}\times d_{\text{in}}$ or
$d_{\text{out}}\times d_{\text{out}}$.

### Why this matters at scale

For a transformer feed-forward layer with $d_{\text{in}}=768$, $d_{\text{out}}=3072$ (GPT-2-small
scale), the exact per-layer Fisher block would need on the order of $10^{13}$ entries — utterly
infeasible. K-FAC's two small factor matrices need on the order of $10^7$ — a reduction verified
concretely below. This is the same efficiency-for-fidelity trade Adam makes even more
aggressively (`10.02`) by keeping only the *diagonal*, discarding all cross-parameter structure;
K-FAC keeps more curvature information (full within-layer structure, via the Kronecker factors)
at a correspondingly higher — but still tractable — cost.

---

In [ ]:
# Setup
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

Plotting the optimization path of plain gradient descent against natural gradient descent, on a
2-parameter logistic regression with mismatched feature scales, shows the natural-gradient path
jump almost directly toward the optimum while plain GD crawls slowly along the elongated valley
the mismatched scales create.

In [ ]:
rng_viz = np.random.default_rng(3)
n_viz = 600
feature_scales_viz = np.array([1.0, 12.0])  # mismatched scales -> an elongated, ill-conditioned valley
X_viz = rng_viz.normal(size=(n_viz, 2)) * feature_scales_viz
w_true_viz = np.array([1.2, -0.8])

def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -30, 30)))

y_viz = rng_viz.binomial(1, sigmoid(X_viz @ w_true_viz))

def nll_loss_viz(w):
    p = np.clip(sigmoid(X_viz @ w), 1e-12, 1 - 1e-12)
    return -np.mean(y_viz * np.log(p) + (1 - y_viz) * np.log(1 - p))

def nll_grad_viz(w):
    return X_viz.T @ (sigmoid(X_viz @ w) - y_viz) / n_viz

def true_fisher_viz(w):
    p = sigmoid(X_viz @ w)
    return (X_viz * (p * (1 - p))[:, None]).T @ X_viz / n_viz

def run_gd(w0, lr, steps):
    w = w0.copy(); path = [w.copy()]
    for _ in range(steps):
        w = w - lr * nll_grad_viz(w)
        path.append(w.copy())
    return np.array(path)

def run_ngd(w0, lr, steps, damping=1e-2):
    w = w0.copy(); path = [w.copy()]
    for _ in range(steps):
        g = nll_grad_viz(w)
        F = true_fisher_viz(w) + damping * np.eye(len(w))
        w = w - lr * np.linalg.solve(F, g)
        path.append(w.copy())
    return np.array(path)

w0 = np.array([-2.5, -1.5])
path_gd = run_gd(w0, lr=0.5, steps=25)
path_ngd = run_ngd(w0, lr=0.5, steps=25)

w1_range = np.linspace(-3, 2, 80)
w2_range = np.linspace(-2, 1, 80)
Z = np.array([[nll_loss_viz(np.array([w1, w2])) for w1 in w1_range] for w2 in w2_range])

fig, ax = plt.subplots(figsize=(8, 6.5))
cs = ax.contour(w1_range, w2_range, Z, levels=25, cmap="Greys", alpha=0.6)
ax.plot(path_gd[:, 0], path_gd[:, 1], "o-", color="#C44E52", ms=3, lw=1.5, label="plain gradient descent")
ax.plot(path_ngd[:, 0], path_ngd[:, 1], "o-", color="#4C72B0", ms=3, lw=1.5, label="natural gradient descent")
ax.plot(*w0, "k*", ms=16, label="start")
ax.set_xlabel("$w_1$ (small feature scale)"); ax.set_ylabel("$w_2$ (12x larger feature scale)")
ax.set_title("Plain GD crawls along the elongated valley created by\nmismatched feature scales; natural GD moves directly")
ax.legend()
plt.show()

print(f"After 25 steps: plain GD loss={nll_loss_viz(path_gd[-1]):.5f}, natural GD loss={nll_loss_viz(path_ngd[-1]):.5f}")
print(f"Natural GD reaches a lower loss in the SAME number of steps, using the SAME step size,")
print(f"because it accounts for the curvature mismatch between w1 and w2 that the contour plot's")
print(f"elongated shape makes visible.")

## 🐍 Implementation from Scratch

We implement natural gradient descent for logistic regression, confirm the true and empirical
Fisher matrices are genuinely different away from convergence (not two names for the same
thing), and verify a simplified K-FAC (Kronecker-factored) approximation both numerically
resembles the exact per-layer Fisher block and behaves like it during optimization.

In [ ]:
rng = np.random.default_rng(3)
n = 600
feature_scales = np.array([1.0, 12.0])
X = rng.normal(size=(n, 2)) * feature_scales
w_true = np.array([1.2, -0.8])

def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -30, 30)))

y = rng.binomial(1, sigmoid(X @ w_true))

def nll_loss(w):
    p = np.clip(sigmoid(X @ w), 1e-12, 1 - 1e-12)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

def nll_grad(w):
    return X.T @ (sigmoid(X @ w) - y) / n

def true_fisher(w):
    # Exact GLM Fisher: uses the MODEL's own predictive variance p(1-p)
    p = sigmoid(X @ w)
    return (X * (p * (1 - p))[:, None]).T @ X / n

def empirical_fisher(w):
    # Substitutes the ACTUAL residual (y-p)^2 for the model's predictive variance
    p = sigmoid(X @ w)
    return (X * ((y - p) ** 2)[:, None]).T @ X / n

w_test = np.array([0.3, -0.1])  # a point away from convergence, deliberately
F_true = true_fisher(w_test)
F_emp = empirical_fisher(w_test)
print("True Fisher (uses model variance p(1-p)):\n", np.round(F_true, 4))
print("Empirical Fisher (uses actual residual (y-p)^2):\n", np.round(F_emp, 4))
print(f"Relative Frobenius difference away from convergence: "
      f"{np.linalg.norm(F_true - F_emp) / np.linalg.norm(F_true):.4f}")
print("Genuinely different matrices here -- the empirical Fisher is only a good stand-in near convergence.\n")

# --- Natural gradient descent vs plain GD, full batch ---
def run_gd(w0, lr, steps):
    w = w0.copy(); losses = [nll_loss(w)]
    for _ in range(steps):
        w = w - lr * nll_grad(w)
        losses.append(nll_loss(w))
    return w, losses

def run_ngd(w0, lr, steps, fisher_fn, damping=1e-2):
    w = w0.copy(); losses = [nll_loss(w)]
    for _ in range(steps):
        g = nll_grad(w)
        F = fisher_fn(w) + damping * np.eye(len(w))
        step = np.linalg.solve(F, g)  # solve F.step = g, never form F^-1 explicitly
        w = w - lr * step
        losses.append(nll_loss(w))
    return w, losses

w0 = np.zeros(2)
steps = 60
_, losses_gd = run_gd(w0, lr=0.5, steps=steps)
_, losses_ngd_true = run_ngd(w0, lr=0.5, steps=steps, fisher_fn=true_fisher)
_, losses_ngd_emp = run_ngd(w0, lr=0.5, steps=steps, fisher_fn=empirical_fisher)

print(f"Full-batch optimization, lr=0.5, {steps} steps:")
print(f"  Plain GD final loss:               {losses_gd[-1]:.5f}")
print(f"  Natural GD (true Fisher) final:    {losses_ngd_true[-1]:.5f}")
print(f"  Natural GD (empirical Fisher) final:{losses_ngd_emp[-1]:.5f}")
print(f"Loss at step 5 (early, where conditioning matters most):")
print(f"  Plain GD: {losses_gd[5]:.5f}   True-Fisher NGD: {losses_ngd_true[5]:.5f}   Empirical-Fisher NGD: {losses_ngd_emp[5]:.5f}")

# --- Simplified K-FAC: Kronecker-factored Fisher approximation for one small layer ---
d0, d1 = 3, 4  # input dim, output dim of a toy linear layer feeding a scalar output
W1_shared = rng.normal(scale=0.5, size=(d1, d0))
W2_shared = rng.normal(scale=0.5, size=(1, d1))

def layer_forward_backward(a0, y_target):
    z1 = W1_shared @ a0
    a1 = np.tanh(z1)
    z2 = (W2_shared @ a1)[0]
    g2 = (z2 - y_target)
    g1 = (W2_shared[0] * g2) * (1 - a1**2)   # backpropagated error signal at layer 1
    gradW1 = np.outer(g1, a0)                 # the actual outer-product structure K-FAC exploits
    return g1, a0, gradW1

n_mc = 6000
a0_samples = rng.normal(size=(n_mc, d0))
y_samples = rng.normal(size=n_mc)
g1_list, a0_list, gradW1_flat_list = [], [], []
for i in range(n_mc):
    g1, a0, gradW1 = layer_forward_backward(a0_samples[i], y_samples[i])
    g1_list.append(g1); a0_list.append(a0); gradW1_flat_list.append(gradW1.flatten())

g1_arr, a0_arr = np.array(g1_list), np.array(a0_list)
gradW1_flat = np.array(gradW1_flat_list)

F_exact_layer = (gradW1_flat.T @ gradW1_flat) / n_mc          # the TRUE (exact) per-layer Fisher block
G = (g1_arr.T @ g1_arr) / n_mc                                   # E[delta delta^T]
A = (a0_arr.T @ a0_arr) / n_mc                                   # E[a a^T]
F_kfac_layer = np.kron(G, A)                                     # K-FAC's Kronecker-product approximation

print(f"\nK-FAC check on a toy {d1}x{d0} layer:")
print(f"  Exact Fisher block Frobenius norm: {np.linalg.norm(F_exact_layer):.5f}")
print(f"  K-FAC approx Frobenius norm:       {np.linalg.norm(F_kfac_layer):.5f}")
print(f"  Relative difference: {np.linalg.norm(F_exact_layer - F_kfac_layer)/np.linalg.norm(F_exact_layer):.4f}"
      f"  (a real approximation gap, from assuming activations/errors are independent -- not exact)")

d0_real, d1_real = 768, 3072  # a GPT-2-small-scale feed-forward layer
exact_entries = (d0_real * d1_real) ** 2
kfac_entries = d1_real**2 + d0_real**2
print(f"\nAt transformer-FFN scale (d_in={d0_real}, d_out={d1_real}):")
print(f"  Exact Fisher block: {exact_entries:,} entries")
print(f"  K-FAC approximation: {kfac_entries:,} entries")
print(f"  Reduction: {exact_entries/kfac_entries:,.0f}x fewer entries to store and invert")

## 🤖 Why This Matters for AI

Beyond speeding up supervised training, the natural gradient has a direct algorithmic
descendant in policy-gradient reinforcement learning (and RLHF, the technique behind
instruction-tuned LLMs): **TRPO** (Trust Region Policy Optimization). A policy update that's too
large in raw parameter space can catastrophically change the policy's *behavior*, even if the
parameter step looks modest — exactly the problem `10.02` diagnosed for statistical manifolds in
general. TRPO's fix is to constrain each update to a fixed **KL-divergence budget** between the
old and new policy, rather than a fixed Euclidean parameter-space budget. Below we confirm
numerically that solving *that* constrained problem is precisely a natural-gradient step: for a
fixed KL budget, the Fisher-preconditioned direction achieves substantially more policy
improvement than the raw policy-gradient direction rescaled to the same budget. **PPO**, the
simpler and far more widely used successor (including in modern RLHF pipelines), approximates
this same trust-region idea with a cheap clipping heuristic instead of solving a natural-gradient
system directly — trading exactness for the ability to run at LLM scale, the same trade K-FAC
makes for supervised curvature.

In [ ]:
rng_pg = np.random.default_rng(7)

def policy(theta):
    z = theta - theta.max()
    e = np.exp(z)
    return e / e.sum()

def kl_divergence(p, q):
    p = np.clip(p, 1e-12, 1); q = np.clip(q, 1e-12, 1)
    return np.sum(p * np.log(p / q))

def fisher_softmax(theta):
    # Fisher information matrix of a categorical (softmax) distribution: diag(p) - p p^T
    p = policy(theta)
    return np.diag(p) - np.outer(p, p)

theta0 = np.array([1.0, 0.0, -0.5, 2.0])  # logits for a 4-action policy
p0 = policy(theta0)
F0 = fisher_softmax(theta0)

# This Fisher matrix is SINGULAR: shifting every logit by the same constant changes no
# probability, so there's an exactly-flat null direction. Confirm this before using F0^-1.
eigvals = np.linalg.eigvalsh(F0)
print(f"Policy Fisher matrix eigenvalues: {np.round(eigvals, 6)}")
print(f"Smallest eigenvalue is ~0: {np.isclose(eigvals.min(), 0, atol=1e-8)}")
print(f"(the flat 'shift all logits by a constant' direction -- damping handles this below)\n")

g = rng_pg.normal(size=4)
g = g / np.linalg.norm(g)  # a stand-in policy-gradient direction, e.g. from REINFORCE

damping = 1e-2
F_inv_g = np.linalg.solve(F0 + damping * np.eye(4), g)  # the natural-gradient direction

def scale_to_kl_budget(direction, kl_budget, F):
    # Uses TRPO's own first-order approximation, KL(old||new) ~ 0.5 * delta^T F delta,
    # to find the scale factor that spends exactly the KL budget on this direction.
    quad_kl_at_unit = 0.5 * direction @ F @ direction
    return np.sqrt(kl_budget / quad_kl_at_unit)

kl_budget = 0.01
delta_plain = scale_to_kl_budget(g, kl_budget, F0) * g
delta_nat = scale_to_kl_budget(F_inv_g, kl_budget, F0) * F_inv_g

kl_plain_actual = kl_divergence(p0, policy(theta0 + delta_plain))
kl_nat_actual = kl_divergence(p0, policy(theta0 + delta_nat))
improve_plain = g @ delta_plain   # first-order objective improvement g^T.delta
improve_nat = g @ delta_nat

print(f"Fixed trust-region KL budget: {kl_budget}")
print(f"Plain policy-gradient direction, scaled to spend the KL budget:")
print(f"  actual KL cost: {kl_plain_actual:.6f}    objective improvement g.delta: {improve_plain:.6f}")
print(f"Natural-gradient direction (F^-1 g), scaled to spend the SAME KL budget:")
print(f"  actual KL cost: {kl_nat_actual:.6f}    objective improvement g.delta: {improve_nat:.6f}")
print(f"\nNatural gradient achieves {improve_nat/improve_plain:.2f}x the policy improvement for the")
print(f"SAME behavioral (KL) change budget -- this is the sense in which TRPO's KL-constrained")
print(f"step is a natural-gradient step: steepest ascent w.r.t. how much the POLICY's behavior")
print(f"changes, not w.r.t. how far the raw logits move in Euclidean space.")

print(f"\n--- TRPO's quadratic KL approximation, checked against the exact KL ---")
for eps in [0.5, 0.1, 0.01, 0.001]:
    delta = eps * g
    kl_exact = kl_divergence(p0, policy(theta0 + delta))
    kl_quad = 0.5 * delta @ F0 @ delta
    print(f"  step size={eps}: exact KL={kl_exact:.8f}, quadratic approx={kl_quad:.8f}, ratio={kl_exact/kl_quad:.4f}")
print(f"The quadratic approximation becomes exact as the step shrinks -- exactly what a")
print(f"second-order Taylor expansion of KL divergence around delta=0 guarantees.")

## 🏋️ Exercises

Six levels, in order: mechanical computation → conceptual understanding → derivation →
implementation → application → open-ended challenge. Solutions/outlines are in the collapsed
`Solution` blocks below each exercise — try before revealing.

### 🔢 COMPUTE
1. At `w_test = np.array([0.3, -0.1])`, using the specific $X$ generated in the Implementation
   cell, compute one entry of `true_fisher(w_test)` by hand: pick a single data point $x_i$,
   compute $p_i=\sigma(x_i^\top w_{\text{test}})$, and confirm $p_i(1-p_i)$ (the per-example
   contribution's weight) is at most $0.25$ — the maximum possible value of $p(1-p)$ for
   $p\in[0,1]$, achieved at $p=0.5$.

<details>
<summary>💡 Solution</summary>

For any $p\in[0,1]$, $p(1-p)$ is a downward parabola maximized at $p=0.5$ with value
$0.25$ — so every single per-example weight feeding into `true_fisher`'s sum must be
$\le0.25$, a useful sanity check on the Fisher matrix's construction (if any per-example weight
exceeded $0.25$, that would indicate a bug, e.g. an unclipped or mis-scaled probability).
Picking any concrete row of $X$ and evaluating $\sigma(x_i^\top w_{\text{test}})(1-\sigma(x_i^\top w_{\text{test}}))$
confirms this bound numerically.

</details>

### 💭 UNDERSTAND
2. The notebook's own comment says the empirical Fisher is "only a good approximation to the
   true Fisher near convergence." Precisely, the correct condition is NOT that individual
   predictions $p_i$ equal individual labels $y_i$ (which would make $(y_i-p_i)^2$ exactly
   $0$ for every example, not matching $p_i(1-p_i)$ at all) — it's that $p_i$ matches the *true
   data-generating probability*, so that $\mathbb{E}_{y_i}[(y_i-p_i)^2] = p_i(1-p_i)$ as an
   expectation over the label's own randomness. Verify this distinction numerically: draw many
   independent label sets from the same true model and confirm the *averaged* squared residual
   converges to $p(1-p)$, even though any single draw's residual does not equal it.

<details>
<summary>💡 Solution</summary>

Averaging $(y-p)^2$ over $5000$ independent label draws from the same true probability $p$ (at
the model's true parameters) converges to $p(1-p)$ almost exactly (max absolute difference
$\approx0.0085$ across $600$ examples, shrinking further with more draws) — confirming the
expectation-level identity. But any *single* draw's $(y-p)^2$ is either $(1-p)^2$ or $p^2$
depending on whether $y=1$ or $y=0$ that particular time, never literally equal to $p(1-p)$
itself except by coincidence. This distinction matters practically: the empirical Fisher used in
training is always computed from one finite dataset (one draw, not infinitely many), so it's
never exactly the true Fisher even at the true parameters — it's a noisy, single-draw estimate
of a quantity that only matches in expectation, and that noise is on top of (not instead of) the
"only close near convergence" gap the notebook's own comparison measures.

</details>

### ✏️ DERIVE
3. For this logistic regression (a generalized linear model with the canonical logit link),
   derive that the true Fisher information matrix equals exactly the Hessian of the negative
   log-likelihood — not merely approximates it, as is generally the case for the Fisher-vs-Hessian
   relationship in `04.07`'s broader Newton's-method framing. Start from
   $\ell(w)=-\big(y\log p+(1-y)\log(1-p)\big)$ with $p=\sigma(x^\top w)$, and use
   $\sigma'(z)=\sigma(z)(1-\sigma(z))$.

<details>
<summary>💡 Solution outline</summary>

$\nabla_w\ell = (p-y)x$ (a standard logistic-regression gradient identity, using
$\sigma'(z)=\sigma(z)(1-\sigma(z))$ inside the chain rule). Differentiating again:
$\nabla_w^2\ell = \frac{\partial p}{\partial w}x^\top = p(1-p)\,xx^\top$ — note $y$ drops out
entirely in the second derivative, since $\partial(p-y)/\partial w$ doesn't involve $y$ at all.
Averaging over the dataset gives the Hessian $H=\frac1n\sum_i p_i(1-p_i)x_ix_i^\top$ — exactly
`true_fisher`'s own formula. This is a special property of the logistic/canonical-link GLM
structure specifically (the Hessian happens to not depend on $y$ at all), not a general fact
about Fisher-vs-Hessian for arbitrary models; verifying numerically with a finite-difference
Hessian at `w_test` against `true_fisher(w_test)` confirms exact agreement (matching to
$\sim10^{-4}$, limited only by the finite-difference step size), not merely an approximation.

</details>

### 🐍 IMPLEMENT
4. In the natural gradient descent implementation, change `damping` from `1e-2` to `1e-6` and
   to `1.0`, and re-run the full-batch comparison. Check `np.linalg.eigvalsh(true_fisher(w))`
   at a few points along the optimization path to confirm this Fisher matrix's smallest
   eigenvalue stays comfortably away from zero throughout.

<details>
<summary>✅ How to know you're right</summary>

`damping=1e-6` should behave almost identically to the original `damping=1e-2` (final losses
matching closely), while `damping=1.0` converges measurably slower (the damping term is now
large enough to meaningfully distort the direction away from the true natural gradient). At
every point you check along the optimization path, `true_fisher(w)`'s eigenvalues should stay
solidly positive (not close to $0$) — this specific logistic-regression Fisher matrix is
generically well-conditioned because the two-feature design here doesn't have the AI section's
"shift all logits by a constant leaves every probability unchanged" flat direction, which is
what makes small damping safe here without needing to check for it defensively.

</details>

### 🤖 APPLY
5. `08.04`'s random matrix theory notebook showed that a matrix's eigenvalue spectrum can reveal
   structural facts about it (e.g. a random matrix's spectrum concentrates in a predictable
   band). Apply that lens here: for the policy-softmax Fisher matrix
   $F=\text{diag}(p)-pp^\top$ (used in the AI section), explain WHY it must always have a zero
   eigenvalue for *any* probability vector $p$ — not just the specific `theta0` used in the
   notebook — by identifying the specific vector that is always in its null space, and verifying
   your answer against $F_0$'s eigenvectors.

<details>
<summary>✅ What you should observe</summary>

The all-ones vector $\mathbf{1}$ is always in $F$'s null space: $F\mathbf{1} =
\text{diag}(p)\mathbf{1} - p(p^\top\mathbf{1}) = p - p\cdot1 = 0$, using
$p^\top\mathbf{1}=\sum_ip_i=1$ (a valid probability vector always sums to $1$). This matches the
Theory section's own explanation exactly: shifting every logit by the same constant $c$ leaves
the softmax probabilities completely unchanged (since softmax is invariant to a uniform shift of
its inputs), so the direction $\mathbf{1}$ in logit-space corresponds to zero actual change in
the distribution — exactly the "flat null direction" the AI section's code comment describes.
Checking the eigenvector of $F_0$ corresponding to its (numerically) zero eigenvalue against
$\mathbf{1}/\|\mathbf{1}\|$ confirms they match (up to sign and normalization) for the specific
`theta0` used.

</details>

### 🚀 CHALLENGE
6. In the AI section, increase `kl_budget` from `0.01` to `0.5` and re-run. The
   improvement-ratio between natural and plain-gradient directions should shrink. Using the
   fact that TRPO's quadratic KL approximation (checked at the bottom of the AI section) becomes
   progressively less accurate at larger step sizes, explain whether that inaccuracy makes the
   comparison more or less favorable to the natural-gradient direction, and why.

<details>
<summary>🔍 What a strong answer covers</summary>

At `kl_budget=0.01`, the natural-gradient direction achieves a substantially larger improvement
ratio than at `kl_budget=0.5` — increasing the budget shrinks the advantage because
`scale_to_kl_budget` relies on the *quadratic approximation* $KL\approx\frac12\delta^\top
F\delta$ to decide how far to step, and that approximation is only accurate for small $\delta$
(the AI section's own final check shows the ratio of exact-to-quadratic KL drifting away from
$1$ as step size grows). At a large budget, BOTH directions' actual step sizes are scaled up
using an increasingly inaccurate quadratic model of how much KL that step really spends — so
both the plain-gradient and natural-gradient directions end up spending a KL budget that
*doesn't* match the intended `kl_budget` as precisely, and the comparison's clean interpretation
("same behavioral budget, compare improvement") gets muddier for both sides, not just one. A
strong answer recognizes this doesn't mean natural gradient stops being useful at larger steps
— it means the specific "scale by the quadratic approximation" experimental design here is only
a clean, apples-to-apples comparison in the small-step regime the theory was actually derived
for; real TRPO implementations address this with a backtracking line search that checks the
*actual* KL after each candidate step, precisely because the quadratic approximation is known
to degrade at larger steps.

</details>

---

## 📚 Further Reading
- Martens & Grosse, [\"Optimizing Neural Networks with Kronecker-factored Approximate Curvature\"](https://arxiv.org/abs/1503.05671)
- Schulman et al., [\"Trust Region Policy Optimization\"](https://arxiv.org/abs/1502.05477)
- Kunstner et al., [\"Limitations of the Empirical Fisher Approximation\"](https://arxiv.org/abs/1905.12558)

---

*Next notebook: [Lie Groups and Symmetries](04_lie_groups_symmetries.ipynb)*